# 🧠 Multi-Head Attention From Scratch (PyTorch)

This notebook implements the foundational component of the Transformer architecture: **Multi-Head Attention**, as introduced in the paper *'Attention Is All You Need'* (Vaswani et al., 2017).

### The Self-Attention Formula
Attention is calculated by mapping a query and a set of key-value pairs to an output:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
Where $d_k$ is the dimensionality of the keys.


## 1. Import Dependencies


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch Version: {torch.__version__}")


## 2. Scaled Dot-Product Attention
This function performs the core Q, K, V attention computations. We include scaling by $1/\sqrt{d_k}$ to prevent gradient saturation, and support an optional mask.


In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    # Get key dimensions
    d_k = q.size(-1)
    
    # Calculate attention scores
    # q: [batch, heads, seq_len, d_k], k^T: [batch, heads, d_k, seq_len]
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Apply optional mask (e.g. for autoregressive causal decoding)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
        
    # Apply softmax to get attention probabilities
    attention_weights = F.softmax(scores, dim=-1)
    
    # Multiply by values
    output = torch.matmul(attention_weights, v)
    return output, attention_weights


## 3. Multi-Head Attention Class
Rather than performing single attention projection, we split our queries, keys, and values into multiple heads to allow the model to jointly attend to information from different representation subspaces at different positions.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Projections for Q, K, V
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        
        # Final output projection
        self.out_linear = nn.Linear(d_model, d_model)
        
    def split_heads(self, x, batch_size):
        # x shape: [batch, seq_len, d_model] -> [batch, seq_len, num_heads, d_k] -> [batch, num_heads, seq_len, d_k]
        x = x.view(batch_size, -1, self.num_heads, self.d_k)
        return x.transpose(1, 2)
        
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # 1. Project Q, K, V
        q = self.q_linear(q)
        k = self.k_linear(k)
        v = self.v_linear(v)
        
        # 2. Split into heads
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)
        
        # 3. Apply Scaled Dot-Product Attention
        output, attention_weights = scaled_dot_product_attention(q, k, v, mask)
        
        # 4. Concatenate heads back
        # output shape: [batch, num_heads, seq_len, d_k] -> [batch, seq_len, num_heads, d_k] -> [batch, seq_len, d_model]
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 5. Final output projection
        return self.out_linear(output), attention_weights


## 4. Verify & Test the Implementation


In [ ]:
# Instantiate Multi-Head Attention module
d_model = 64
num_heads = 8
mha = MultiHeadAttention(d_model, num_heads)

# Create mock batch input: [batch_size=2, seq_len=5, d_model=64]
x = torch.randn(2, 5, d_model)

# Forward pass
output, attn_weights = mha(x, x, x)

print("Execution Successful!")
print(f"Input Shape:  {x.shape}")
print(f"Output Shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
